# Imitation Learning: Teaching a Robot by Example

In this lab you will train a robot to run by **watching an expert** — no hand-coded rules, no reward engineering. The technique is called **Behavior Cloning**:

1. An expert policy runs the robot; we record what it sees and does.
2. We treat those recordings as a supervised dataset: **input = observation, label = action**.
3. We train a small neural network (MLP) to predict the expert's actions.
4. We deploy our learned policy and compare it to the expert.

The robot is **HalfCheetah** from MuJoCo — a 2D simulated cheetah that learns to run forward.

---

## ⚙️ Setup — read this before running anything

### 💻 Running locally

Do this **once** in a terminal:
```bash
conda create -n il-lab python=3.11 -y
conda activate il-lab
pip install -r requirements.txt
```
Then launch Jupyter from inside the repo folder with that environment active. **Skip Step 0.**

### ☁️ Running on Google Colab

**1. Switch to a GPU runtime:**
> Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save

**2. Run Step 0 first** and wait for `✅ Setup complete`.

> ⚠️ **Steps 1a, 1b, and 7 (video cells) do not work on Colab** — skip those. All other steps work fine.

---

## Step 0 — Colab Setup ☁️

> **Local users: skip this cell.**  
> **Colab users: run this cell first** and wait for `✅ Setup complete`.

In [ ]:
# ════════════════════════════════════════════════
# COLAB ONLY — local users skip this cell
# ════════════════════════════════════════════════
import subprocess, sys, os

REPO = 'pgss-L4R-IL-public'
if not os.path.isdir(REPO):
    r = subprocess.run(['git', 'clone', 'https://github.com/alokshah04/pgss-L4R-IL-public.git'],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    print('Repo already cloned.')
if not os.getcwd().endswith(REPO):
    os.chdir(REPO)
print('Working directory:', os.getcwd())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('\n✅ Setup complete. Continue to Step 1.')

---
## Step 1 — Imports and Expert 💻☁️

*(You may see a `Could not deserialize lr_schedule` warning — harmless. The policy weights load fine.)*

In [ ]:
import gymnasium as gym
from stable_baselines3 import SAC
import imageio
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.display import Video, display

ENV_NAME    = 'HalfCheetah-v5'
EXPERT_PATH = 'experts/HalfCheetah-v5'

expert = SAC.load(EXPERT_PATH)
print('Expert policy loaded.')

env = gym.make(ENV_NAME)
print(f'Observation space: {env.observation_space}')   # what the robot sees
print(f'Action space:      {env.action_space}')        # what the robot does
env.close()

### Step 1a — Video recorder 💻 (local only)

> **Colab users: skip Steps 1a and 1b.** Video rendering requires a display, which Colab does not have.

We use gymnasium's built-in `rgb_array` renderer to record frames.

In [ ]:
# ════════════════════════════════════════════════
# LOCAL ONLY — Colab users skip this cell
# ════════════════════════════════════════════════
def record_policy(step_fn, env_name, path, max_steps=300, seed=1, fps=30):
    """Run step_fn(obs) -> action and save an MP4 using gymnasium's rgb_array renderer."""
    env = gym.make(env_name, render_mode='rgb_array')
    obs, _ = env.reset(seed=seed)
    frames, total_reward = [], 0.0
    for _ in range(max_steps):
        action = step_fn(obs)
        obs, reward, done, truncated, _ = env.step(action)
        total_reward += reward
        frames.append(env.render())
        if done or truncated:
            break
    env.close()
    imageio.mimsave(path, frames, fps=fps)
    print(f'Saved {path}  |  reward: {total_reward:.1f}  |  frames: {len(frames)}')
    return total_reward

print('record_policy defined.')

### Step 1b — Watch the expert run 💻 (local only)

> **Colab users: skip this cell.**

In [ ]:
# ════════════════════════════════════════════════
# LOCAL ONLY — Colab users skip this cell
# ════════════════════════════════════════════════
def expert_step(obs):
    action, _ = expert.predict(obs, deterministic=True)
    return action

record_policy(expert_step, ENV_NAME, 'expert_demo.mp4')
display(Video('expert_demo.mp4', embed=True, width=600))

---
## Step 2 — Collect Demonstrations 💻☁️

We roll out the expert many times and store every *(observation, action)* pair it produces.

Each **trajectory** is a dictionary with three lists:
```
{
  'obs'     : [o_0, o_1, ..., o_T],   # what the robot saw at each step
  'acts'    : [a_0, a_1, ..., a_T],   # what the expert did at each step
  'rewards' : [r_0, r_1, ..., r_T],   # reward received at each step
}
```

We collect `NUM_TRAJ` trajectories in total, then split them **70 / 10** into a **training set** and a **validation set**. The model only sees the training set during gradient updates; the validation set is used to check that it generalises to unseen data.

In [ ]:
NUM_TRAJ      = 80    # ← try changing this!
MAX_TIMESTEPS = 300
VAL_TRAJ      = 10   # held-out trajectories for validation

print(f'Collecting {NUM_TRAJ} expert trajectories...')

env = gym.make(ENV_NAME)
policy = SAC.load(EXPERT_PATH)
env.reset(seed=42)

all_trajectories = []
for ep in range(NUM_TRAJ):
    obs, _ = env.reset()
    traj = {'obs': [], 'acts': [], 'rewards': []}
    for _ in range(MAX_TIMESTEPS):
        traj['obs'].append(obs.copy())
        action, _ = policy.predict(obs, deterministic=True)
        traj['acts'].append(action)
        obs, reward, done, truncated, _ = env.step(action)
        traj['rewards'].append(reward)
        if done or truncated:
            break
    all_trajectories.append(traj)
env.close()

# Split into train and val
train_trajs = all_trajectories[:NUM_TRAJ - VAL_TRAJ]
val_trajs   = all_trajectories[NUM_TRAJ - VAL_TRAJ:]

print(f'Train trajectories: {len(train_trajs)}')
print(f'Val   trajectories: {len(val_trajs)}')

In [ ]:
# Inspect one trajectory
traj0 = train_trajs[0]
obs0, acts0 = np.array(traj0['obs']), np.array(traj0['acts'])
print(f'Trajectory length: {len(traj0["obs"])} steps')
print(f'Observation shape: {obs0.shape}  (T x obs_dim)')
print(f'Action shape:      {acts0.shape}  (T x act_dim)')
print(f'Total reward:      {sum(traj0["rewards"]):.1f}')
print('\nFirst observation (what the robot sees at t=0):')
print(obs0[0])
print('\nFirst expert action (what the expert does at t=0):')
print(acts0[0])

---
## Step 3 — Build a Supervised Learning Dataset 💻☁️

We flatten all trajectories into a flat list of *(observation, action)* pairs — exactly like any supervised learning problem:
- **Input X** = observation vector (17 numbers: joint angles & velocities)
- **Label Y** = action vector (6 numbers: one torque per joint)

We do this separately for the train and validation splits, then wrap each in a PyTorch `DataLoader` that handles batching and shuffling.

In [ ]:
class TrajDataset(Dataset):
    """
    Converts a list of trajectories into a PyTorch Dataset of (obs, action) pairs.
    Each sample is one timestep: the observation the robot saw and the action the expert took.
    """
    def __init__(self, trajectories):
        obs_list, act_list = [], []
        for traj in trajectories:
            obs_list.extend(traj['obs'])
            act_list.extend(traj['acts'])
        self.inputs = torch.tensor(np.array(obs_list), dtype=torch.float32)
        self.labels = torch.tensor(np.array(act_list), dtype=torch.float32)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.labels[idx]


BATCH_SIZE = 256

train_dataset = TrajDataset(train_trajs)
val_dataset   = TrajDataset(val_trajs)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print(f'Train samples: {len(train_dataset)}')
print(f'Val   samples: {len(val_dataset)}')

obs_batch, act_batch = next(iter(train_loader))
print(f'\nOne batch: observations {obs_batch.shape}, actions {act_batch.shape}')

OBS_DIM = obs_batch.shape[1]
ACT_DIM = act_batch.shape[1]
print(f'obs_dim = {OBS_DIM},  act_dim = {ACT_DIM}')

---
## Step 4 — Define the Policy Network 💻☁️

Our learned policy is a **Multi-Layer Perceptron (MLP)**. It takes an observation as input and outputs a predicted action.

Each hidden layer has three components:
- **`Linear`** — a learned matrix multiply (the weights we train)
- **`LayerNorm`** — normalises activations so the policy generalises better to observations it didn't see during training
- **`GELU`** — a smooth non-linearity that lets the network represent complex functions

```
observation (17) -> [Linear -> LayerNorm -> GELU] x 3 -> action (6)
```

The output layer has **no activation** — actions are continuous real numbers with no fixed range (before environment clipping).

**Your turn — try changing `HIDDEN_SIZE` (e.g. 64, 128, 512).**

In [ ]:
class MLP(nn.Module):
    """
    A 3-hidden-layer MLP that maps observations to actions.
    This is our learned policy: given what the robot sees, predict what to do.
    """
    def __init__(self, obs_dim, act_dim, hidden_size=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, act_dim),
        )

    def forward(self, x):
        return self.net(x)


HIDDEN_SIZE = 256   # ← try changing this!
model = MLP(obs_dim=OBS_DIM, act_dim=ACT_DIM, hidden_size=HIDDEN_SIZE)
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

---
## Step 5 — Train the Policy 💻☁️

We train with **Mean Squared Error** loss — for each observation in the batch, the model predicts an action and we penalise it for how far that prediction is from what the expert actually did:

$$\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} \| \hat{a}_i - a_i^{\text{expert}} \|^2$$

We use the **AdamW** optimiser with a **cosine learning-rate schedule** that gradually reduces the learning rate over training.

After each epoch we also compute the **validation loss** on the held-out trajectories — this tells us whether the model is generalising or just memorising the training data.

**On Colab with T4 GPU:** ~2 minutes &nbsp;|&nbsp; **On a laptop CPU:** ~10 minutes

**Your turn — try changing `EPOCHS` or `LR`.**

In [ ]:
EPOCHS = 1000   # ← try changing this!
LR     = 1e-3   # ← or this!
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {DEVICE}')

model = model.to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.MSELoss()

train_losses, val_losses = [], []

for epoch in tqdm(range(EPOCHS), desc='Training'):
    # ── training pass ────────────────────────────────────────
    model.train()
    total_train = 0.0
    for obs, acts in train_loader:
        obs, acts = obs.to(DEVICE), acts.to(DEVICE)
        preds = model(obs)
        loss  = criterion(preds, acts)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_train += loss.item() * obs.size(0)
    scheduler.step()
    train_losses.append(total_train / len(train_dataset))

    # ── validation pass (no gradients) ───────────────────────
    model.eval()
    total_val = 0.0
    with torch.no_grad():
        for obs, acts in val_loader:
            obs, acts = obs.to(DEVICE), acts.to(DEVICE)
            preds = model(obs)
            total_val += criterion(preds, acts).item() * obs.size(0)
    val_losses.append(total_val / len(val_dataset))

print(f'Final train loss: {train_losses[-1]:.6f}')
print(f'Final val   loss: {val_losses[-1]:.6f}')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train loss')
plt.plot(val_losses,   label='Val loss', linestyle='--')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.tight_layout(); plt.savefig('training_loss.png', dpi=120); plt.show()

---
## Step 6 — Evaluate on Test Rollouts 💻☁️

A low validation loss means the model predicts expert actions accurately on unseen data — but that's not the same as the robot actually being able to run. When deployed, small prediction errors **compound** over time: the robot ends up in states it never saw during training, which leads to worse predictions, which leads to worse states, and so on.

The true test is to **roll the learned policy out in the environment** and measure the total reward it earns. We compare this against the expert over `NUM_EVAL_TRAJ` fresh episodes (the **test set**).

In [ ]:
NUM_EVAL_TRAJ = 10
model = model.cpu().eval()

expert_returns  = []
learned_returns = []

rng = np.random.default_rng(0)
for _ in tqdm(range(NUM_EVAL_TRAJ), desc='Evaluating'):
    seed = int(rng.integers(0, 2**31))

    # ── expert rollout ───────────────────────────────────────
    env = gym.make(ENV_NAME)
    obs, _ = env.reset(seed=seed)
    ep_reward = 0.0
    for _ in range(MAX_TIMESTEPS):
        action, _ = policy.predict(obs, deterministic=True)
        obs, reward, done, truncated, _ = env.step(action)
        ep_reward += reward
        if done or truncated: break
    env.close()
    expert_returns.append(ep_reward)

    # ── learned policy rollout ───────────────────────────────
    env = gym.make(ENV_NAME)
    obs, _ = env.reset(seed=seed)
    ep_reward = 0.0
    with torch.no_grad():
        for _ in range(MAX_TIMESTEPS):
            obs_t  = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
            action = model(obs_t).squeeze(0).numpy()
            obs, reward, done, truncated, _ = env.step(action)
            ep_reward += reward
            if done or truncated: break
    env.close()
    learned_returns.append(ep_reward)

expert_returns  = np.array(expert_returns)
learned_returns = np.array(learned_returns)

print(f'Expert  avg return: {expert_returns.mean():.1f} +/- {expert_returns.std():.1f}')
print(f'Learned avg return: {learned_returns.mean():.1f} +/- {learned_returns.std():.1f}')
reward_gap = (expert_returns.mean() - learned_returns.mean()) / abs(expert_returns.mean())
print(f'Normalised reward gap: {reward_gap:.1%}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([expert_returns, learned_returns],
           labels=['Expert', 'Learned (ours)'], patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6))
ax.set_ylabel('Total Episode Reward'); ax.set_title('Expert vs Learned Policy — Test Rollouts')
plt.tight_layout(); plt.savefig('eval_rewards.png', dpi=120); plt.show()

---
## Step 7 — Watch Your Robot Run! 💻 (local only)

> **Colab users: skip this step.**

In [ ]:
# ════════════════════════════════════════════════
# LOCAL ONLY — Colab users skip this cell
# ════════════════════════════════════════════════
model.eval()

def learned_step(obs):
    with torch.no_grad():
        obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
        return model(obs_t).squeeze(0).numpy()

print('--- Expert ---')
record_policy(expert_step, ENV_NAME, 'expert_demo.mp4')
display(Video('expert_demo.mp4', embed=True, width=600))

print('\n--- Learned policy ---')
record_policy(learned_step, ENV_NAME, 'learned_policy.mp4')
display(Video('learned_policy.mp4', embed=True, width=600))

---
## Discussion Questions

1. **Data matters.** Try `NUM_TRAJ = 20` vs `80`. How does the amount of demonstration data affect validation loss and final reward? Why?
2. **Train vs val loss.** Do the two curves track each other closely? What would it mean if val loss stayed high while train loss dropped?
3. **Compounding error.** The model can have near-zero validation loss yet still earn much lower reward than the expert. Why? *(Hint: at test time the robot visits states it never saw during training.)*
4. **Architecture.** Try `HIDDEN_SIZE = 32` vs `512`. How does model capacity affect train loss, val loss, and reward?
5. **Epochs.** At what point does training more stop helping? What does the val loss curve tell you?
6. **Beyond imitation.** What information does the expert have that we are not giving our policy? How might you use the reward signal to improve further?

---
## A Note on Seed Variance

You may notice that running the same code twice produces noticeably different results — sometimes the robot runs well, sometimes it falls, even with identical hyperparameters. This is normal and worth understanding.

There are two independent sources of randomness in this pipeline:

**1. Training randomness** — the initial network weights are randomly initialised, and minibatches are drawn in a random order each epoch. Two training runs with different seeds can converge to different local minima, producing policies with meaningfully different deployment behaviour.

**2. Evaluation randomness** — each test episode starts from a different random initial state. A policy that averages well over many seeds may still fail on specific unlucky ones.

This means a single reward number is not a reliable measure of how good your policy is. When comparing two approaches (e.g. for Bonus 2), you should:
- Run evaluation over at least 10 episodes and report the **mean ± std**
- Consider re-running training 2–3 times with different seeds and averaging the results

If you see a large standard deviation in the box plot from Step 6, that is the compounding error problem manifesting as high variance across starting conditions — not a bug in your code.

---
## Bonus 1 — Save and Reload Your Policy 💻☁️

In [ ]:
torch.save(model.state_dict(), 'my_policy.pt')
print('Policy saved to my_policy.pt')

loaded_model = MLP(obs_dim=OBS_DIM, act_dim=ACT_DIM, hidden_size=HIDDEN_SIZE)
loaded_model.load_state_dict(torch.load('my_policy.pt', map_location='cpu'))
loaded_model.eval()
print('Policy reloaded successfully.')

---
## Bonus 2 — Beat the Baseline 💻☁️

The default recipe (3-layer MLP, LayerNorm, GELU, AdamW, cosine schedule) is a strong starting point, but it is not the ceiling. Can you find a configuration that achieves a higher terminal reward?

You are free to modify anything: the architecture, the optimizer, the training pipeline, or the data collection strategy. Every change should be supported by a resource you found — paper, blog post, documentation, or other credible source.

Some directions worth exploring:

- **Architecture:** deeper or wider networks, residual/skip connections, different normalization (BatchNorm, RMSNorm), different activations (SiLU, Mish)
- **Optimizer:** learning rate warmup, gradient clipping (`torch.nn.utils.clip_grad_norm_`), different weight decay
- **Regularization:** dropout, early stopping on val loss
- **Data:** per-feature observation normalization (subtract mean, divide by std), more trajectories, longer horizons
- **Objective:** L1 loss, Huber loss, or weighting samples by trajectory quality

Define your bonus model with a new variable name (e.g. `bonus_model`) so it does not overwrite the baseline, and save it to `my_policy_bonus.pt`.

In [ ]:
# ── Bonus 2 sandbox ──────────────────────────────────────────────────────────
# Modify whatever you like below. Keep the baseline model above untouched.

# Example: observation normalization (compute from training data)
obs_mean = train_dataset.inputs.mean(dim=0)
obs_std  = train_dataset.inputs.std(dim=0).clamp(min=1e-8)

# To use normalization, wrap your model's forward pass:
# normalized_obs = (obs - obs_mean) / obs_std
# then pass normalized_obs through the network.

# Define your improved model here, train it, evaluate it, then save:
# torch.save(bonus_model.state_dict(), 'my_policy_bonus.pt')
# print('Bonus policy saved.')